In [8]:
import torch
import torch.nn as nn
from einops import rearrange, einsum

# Problem 1: 实现 linear 模块 (1 分)
class Linear(torch.nn.Module):
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__() # 调用父类初始化
        '''
        in_features: int  输入特征维度
        out_features: int  输出特征维度
        device: torch.device | None = None  参数存放在哪个设备上
        dtype: torch.dtype | None = None  参数的数据类型
        '''

        # torch.empty() 只分配内存，不初始化数值；用 torch.zeros, torch.ones, torch.randn 创建张量都可以，只是会做一次无意义的填充
        self.w = nn.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype)) # nn.Linear()
        

        mean, std = 0, 2/(in_features+out_features) ** 0.5
        nn.init.trunc_normal_(self.w, mean=mean, std=std, a=-3*std, b=3*std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        对输出应用线性变换 y = xA^T
        x: [..., in_features] 任意数量的 batch 维度的输入特征
        return: [..., out_features] 线性变换后的输出
        '''
        # einsum 实现矩阵乘法，等价于 y = x @ self.w.T 或者是 y = torch.matmul(x, self.w.T)
        y = einsum(x, self.w, "... d_in, d_out d_in -> ... d_out") # 对 d_in 维度执行矩阵变换，并将该操作广播到 d_in 维度前面的所有维度上
        return y


In [9]:
m = Linear(256, 512) # 初始化
input = torch.randn(32, 128, 256) # 随机创建指定形状的 tensor
output = m(input) # 调用 linear.forward() 方法进行前向传播，形状为 (32, 128, 512)
print(f"output 形状：{output.shape}")

output 形状：torch.Size([32, 128, 512])


In [10]:
# Problem 2: 实现 Embedding 模块 (1 分)
class Embedding(torch.nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, device=None, dtype=None):
        super().__init__()
        '''
        num_embeddings: int  词表大小，取值范围 [0, num_embeddings‑1]
        embedding_dim: int  词向量的维度，也即模型特征维度 d_model
        device: torch.device | None = None  参数存放在哪个设备上
        dtype: torch.dtype | None = None  参数的数据类型
        '''

        self.embed = nn.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))

        # 截断正态初始化，均值 0，标准差 1，截断区间 [-3,3]
        nn.init.trunc_normal_(self.embed, mean=0, std=1, a=-3, b=3)


    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        '''
        根据 token_ids 查表取向量
        token_ids: [batch_size, seq_len]
        return: [batch_size, seq_len, embed_dim]
        '''

        # pytorch 索引操作，等价于 torch.index_select
        return self.embed[token_ids] # 用 token_ids 中的每个元素去索引 self.embed 的第 0 维，形状变化为：(batch_size, seq_len) -> (batch_size, seq_len, embed_dim)
        # id = 0 -> self.embed[0]


In [11]:
# 使用方式
emb = Embedding(num_embeddings=10000, embedding_dim=512) # 初始化词表大小 10000，向量维度 512 的 embedding 层
token_ids = torch.tensor([[1, 42, 99], [7, 13, 0]]) # 形状为 (2, 3)，元素是词表索引
output = emb(token_ids) # 根据索引查表，输出 (2, 3, 512)
print(f"output 形状：{output.shape}")

output 形状：torch.Size([2, 3, 512])
